<div dir="rtl">

# ۰۰ — آماده‌سازی محیط



</div>

<div dir="rtl">

## یک نکته‌ی مهم درباره‌ی نسخه‌ها

- SDK رسمی پایتون MCP دو خط نسخه دارد: **1.x** و **2.x**. در نسخه‌ی 2، کلاس `FastMCP` به `MCPServer` تغییر نام داده است.
- کتابخانه‌ی `langchain-mcp-adapters` (که در نوت‌بوک‌های ۰۶ و ۰۷ لازم داریم) فعلاً فقط با **mcp 1.x** کار می‌کند.
- به همین دلیل کل دوره روی `mcp>=1.24,<2` نوشته شده است. این نسخه پروتکل **2025-11-25** را پیاده می‌کند؛ همان چیزی که در اسلایدها دیدیم (`initialize`، session و ...).
- نسخه‌ی جدید مشخصات MCP (**2026-07-28**) پروتکل را **بدون حالت (stateless)** کرده است: دیگر `initialize` و `Mcp-Session-Id` وجود ندارد. هر جا که این تغییر مهم باشد، در خود نوت‌بوک توضیح داده‌ایم.

اگر بعداً LangChain از mcp 2.x پشتیبانی کرد، مهاجرت بیشتر در حد تغییر نام‌هاست:

</div>

```python
# mcp 1.x (this course)
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("Demo")

# mcp 2.x
from mcp.server.mcpserver import MCPServer
mcp = MCPServer("Demo")
```

<div dir="rtl">

## ۱. ساخت محیط مجازی

در پوشه‌ی همین نوت‌بوک‌ها یک ترمینال باز کنید.

**ویندوز (PowerShell):**

</div>

```powershell
python -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
pip install -r requirements.txt
```

<div dir="rtl">

اگر PowerShell اجازه‌ی اجرای اسکریپت را نداد، یک بار این دستور را بزنید:
`Set-ExecutionPolicy -Scope CurrentUser RemoteSigned`

**لینوکس / مک:**

</div>

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
pip install -r requirements.txt
```

<div dir="rtl">

بعد Jupyter را **از داخل همین محیط** اجرا کنید (`jupyter lab` یا `jupyter notebook`)، یا در VS Code همین `.venv` را به‌عنوان kernel انتخاب کنید.

## ۲. نصب Node.js و uv

بعضی سرورهای آماده‌ی MCP با `npx` (از Node.js) و بعضی با `uvx` (از uv) اجرا می‌شوند. ابزار **MCP Inspector** هم با `npx` اجرا می‌شود.

**ویندوز:**

</div>

```powershell
winget install OpenJS.NodeJS.LTS
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

<div dir="rtl">

بعد از نصب، **ترمینال و Jupyter را ببندید و دوباره باز کنید** تا `PATH` جدید خوانده شود. اگر `uvx` پیدا نشد، مسیر `C:\Users\<USERNAME>\.local\bin` را به متغیر محیطی `PATH` اضافه کنید.

**لینوکس:**

</div>

```bash
# Node.js (Ubuntu/Debian) - or use nvm
sudo apt install -y nodejs npm
# uv / uvx
curl -LsSf https://astral.sh/uv/install.sh | sh
```

<div dir="rtl">

### دانلود اولیه‌ی سرورهای آماده

در نوت‌بوک ۰۶ از دو سرور آماده استفاده می‌کنیم. بهتر است یک بار آن‌ها را **در ترمینال** اجرا کنید تا دانلود شوند. اگر از پروکسی یا VPN استفاده می‌کنید، این کار مهم‌تر است (توضیحش در نوت‌بوک ۰۶ آمده). هر دستور را اجرا کنید و وقتی پیام اجرای سرور را دیدید، `Ctrl+C` بزنید:

</div>

```bash
uvx mcp-server-time
npx -y @modelcontextprotocol/server-filesystem .
```

<div dir="rtl">

## ۳. فایل `.env`

فایل `.env.example` را کپی کنید، نامش را `.env` بگذارید و کلید OpenAI خود را در آن بنویسید:

</div>

```
OPENAI_API_KEY=sk-...
```

<div dir="rtl">

در ابتدای همه‌ی نوت‌بوک‌ها `load_dotenv()` را صدا می‌زنیم تا این مقدار به‌صورت متغیر محیطی در دسترس باشد. کتابخانه‌ی `openai` و LangChain کلید را خودشان از متغیر محیطی `OPENAI_API_KEY` می‌خوانند.

## ۴. فایل کمکی `mcp_helpers.py`

این فایل کنار نوت‌بوک‌ها قرار دارد و چند تابع کوچک دارد:

- `setup_notebook()`: دو مشکل رایج Jupyter روی ویندوز را برطرف می‌کند (event loop و `stderr`) تا سرورهای stdio درست اجرا شوند.
- `start_server()` و `stop_server()`: یک سرور HTTP را در پس‌زمینه بالا می‌آورد و می‌بندد، تا مجبور نباشید برای هر مثال یک ترمینال جدا باز کنید. خروجی سرور در پوشه‌ی `logs` ذخیره می‌شود.
- `parse_sse()` و `show_json()`: برای خواندن و نمایش پیام‌های خام.

حالا همه‌چیز را بررسی کنیم:

</div>

In [1]:
from dotenv import load_dotenv
load_dotenv()  # reads OPENAI_API_KEY (and other settings) from the .env file

from mcp_helpers import setup_notebook, start_server, stop_server, parse_sse, show_json
setup_notebook()

Notebook ready | Python 3.12.12 | win32


In [3]:
import os, shutil, importlib.metadata as md

print("OPENAI_API_KEY found:", bool(os.getenv("OPENAI_API_KEY")))
for pkg in ["mcp", "langchain-mcp-adapters", "langchain", "langchain-openai", "openai", "httpx", "arxiv"]:
    try:
        print(f"{pkg:25s}", md.version(pkg))
    except md.PackageNotFoundError:
        print(f"{pkg:25s}", "NOT INSTALLED")
for tool in ["node", "npx", "uvx"]:
    print(f"{tool:25s}", shutil.which(tool) or "NOT FOUND")

OPENAI_API_KEY found: True
mcp                       1.25.0
langchain-mcp-adapters    0.3.0
langchain                 1.3.1
langchain-openai          1.2.1
openai                    2.37.0
httpx                     0.28.1
arxiv                     4.0.0
node                      C:\Program Files\nodejs\node.EXE
npx                       C:\Program Files\nodejs\npx.CMD
uvx                       C:\Users\Alireza\AppData\Local\com.LangflowDesktop\uv\uvx.EXE


<div dir="rtl">

همه‌ی موارد بالا باید پیدا شوند و نسخه‌ی `mcp` باید با `1.` شروع شود.

## ۵. تست کلید OpenAI

در تمام دوره از یک مدل استفاده می‌کنیم. اگر خواستید مدل را عوض کنید، فقط مقدار `MODEL` را تغییر دهید.

</div>

In [5]:
from openai import OpenAI

MODEL = "gpt-5.2"
client = OpenAI()  # uses OPENAI_API_KEY from the environment

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "در یک جمله‌ی کوتاه بگو MCP (Model Context Protocol) چیست."}],
)
print(response.choices[0].message.content)

MCP (Model Context Protocol) یک استاندارد برای وصل‌کردن مدل‌های هوش مصنوعی به ابزارها، سرویس‌ها و منابع داده‌ی بیرونی به‌صورت ساخت‌یافته و امن است.


<div dir="rtl">

## رفع اشکال‌های رایج

| مشکل | راه‌حل |
|---|---|
| `Port 8000 is already in use` | سرور قبلی هنوز روشن است. kernel را restart کنید، یا پروسه را ببندید (دستورها پایین آمده‌اند) |
| `uvx` یا `npx` پیدا نمی‌شود | بعد از نصب، ترمینال و Jupyter را کامل ببندید و دوباره باز کنید |
| اولین اجرای `npx` یا `uvx` خیلی طول می‌کشد یا `Connection closed` می‌دهد | بار اول بسته دانلود می‌شود. یک بار دیگر سلول را اجرا کنید |
| `ModuleNotFoundError: mcp.server.fastmcp` | mcp نسخه‌ی 2 نصب شده است. دستور `pip install "mcp<2"` را بزنید |
| `NameError: name 'tools' is not defined` | سلول قبلی خطا داده است. خروجی آن سلول را بالاتر ببینید؛ معمولاً `Connection closed` است، یعنی سرور اجرا نشد |
| `Connection closed` برای سرورهای `uvx` یا `npx` پشت پروکسی | سرور stdio متغیرهای پروکسی را به ارث نمی‌برد. راه‌حل در نوت‌بوک ۰۶ (کلید `env`) آمده است |

</div>

```powershell
# Windows: who is using port 8000?
netstat -ano | findstr :8000
taskkill /PID <PID> /F
```
```bash
# Linux / macOS
lsof -i :8000
kill <PID>
```